In [0]:
# basic_manual_etl_pyspark.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, current_timestamp

# -----------------------------
# 1. Create Spark Session
# -----------------------------
spark = (
    SparkSession.builder
    .appName("ManualETLPipeline")
    .getOrCreate()
)

# -----------------------------
# 2. Extract
# Create data manually
# -----------------------------
data = [
    (1, "john doe", "john@example.com", 25),
    (2, "jane smith", "jane@example.com", 17),
    (3, "bob lee", "bob@example.com", 31),
]

columns = ["id", "name", "email", "age"]

df = spark.createDataFrame(data, columns)

print("Raw Data:")
df.show()

# -----------------------------
# 3. Transform
# -----------------------------
transformed_df = (
    df
    .filter(col("age") >= 18)
    .withColumn("name", upper(col("name")))
    .withColumn("processed_at", current_timestamp())
)

print("Transformed Data:")
transformed_df.show()

# -----------------------------
# 4. Load
# Save as parquet
# -----------------------------
output_path = "/Volumes/workspace/default/volume2"

(
    transformed_df.write
    .mode("overwrite")
    .parquet(output_path)
)

print(f"ETL completed. Output saved to: {output_path}")

# -----------------------------
# 5. Stop Spark Session
# -----------------------------
spark.stop()

Raw Data:
+---+----------+----------------+---+
| id|      name|           email|age|
+---+----------+----------------+---+
|  1|  john doe|john@example.com| 25|
|  2|jane smith|jane@example.com| 17|
|  3|   bob lee| bob@example.com| 31|
+---+----------+----------------+---+

Transformed Data:
+---+--------+----------------+---+--------------------+
| id|    name|           email|age|        processed_at|
+---+--------+----------------+---+--------------------+
|  1|JOHN DOE|john@example.com| 25|2026-05-20 01:01:...|
|  3| BOB LEE| bob@example.com| 31|2026-05-20 01:01:...|
+---+--------+----------------+---+--------------------+

ETL completed. Output saved to: /Volumes/workspace/default/volume2
